# 01 · Train và test SMLP4Rec trên booking Expedia

Lượt chạy kiểm chứng pipeline của tuần 3, được viết lại thành notebook tường minh. Thay thế `scripts/run_smlprec_expedia.py`.

- **Dữ liệu**: `data/interim/recbole/expedia/expedia.inter` (chỉ booking, item = `hotel_cluster`, proxy cho loại phòng / gói), được tạo bởi `scripts/expedia_to_recbole.py`.
- **Chia tập**: chia theo thời gian toàn cục 80/10/10 của RecBole trên các mục tiêu next-booking. Booking đầu tiên của mỗi user không bao giờ là mục tiêu, nên user cold không được đánh giá ở đây (xem notebook 03).
- **Mô hình**: SMLP4Rec, `src/models/smlprec.py` (bản điều chỉnh nghiên cứu từ fork, đầu vào chỉ có lịch sử), config `configs/smlprec_expedia.yaml` (3 epoch, config nhẹ, seed 2022).
- **Phần được viết tường minh ở đây**: vòng lặp huấn luyện, chấm điểm, tính metric và chọn epoch tốt nhất. RecBole chỉ dùng để dựng dataset/dataloader và cho lớp mô hình.
- **Metric**: Recall@K và NDCG@K, K = 5, 10, 20, xếp hạng đầy đủ trên 100 cluster (`nbp.eval.metrics`, đã có unit test).
- **Kiểm tra tái lập**: mục cuối so sánh mọi con số với lượt chạy của script, `results/week3_implementation/smlprec_expedia_run.json`.

Kernel: `smlp4rec` (venv Python 3.11 với RecBole 1.0.1, numpy 1.23, pandas 1.5). Output ghi vào `experiments/<date>_expedia_smlp4rec_week3-repro/` (gitignored): trọng số từng epoch, checkpoint tốt nhất, `metrics.json`.

In [1]:
import functools
import json
import logging
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # import torch before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

# RecBole 1.0.1 checkpoints store the Config object; PyTorch >= 2.6 defaults to weights_only=True.
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.compare import compare_nested
from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)  # recbole's sine.py turns anomaly detection on at import
logging.disable(logging.INFO)  # RecBole's dataset logging

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
REFERENCE = ROOT / "results" / "week3_implementation" / "smlprec_expedia_run.json"
TAG = "week3-repro"
RUN_DIR = ROOT / "experiments" / f"{date.today():%Y-%m-%d}_expedia_smlp4rec_{TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-05_expedia_smlp4rec_week3-repro


## 1. Config và dữ liệu

`init_seed` chạy trước khi dựng dataset và mô hình, theo cùng thứ tự với script, nên khởi tạo mô hình và việc xáo trộn theo từng epoch dùng cùng các số ngẫu nhiên.

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia",
    config_file_list=[str(CONFIG)],
    config_dict={"data_path": str(WORK), "checkpoint_dir": str(RUN_DIR)},
)
init_seed(config["seed"], config["reproducibility"])

t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
t_data = time.time() - t0

ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
IID, TIME = config["ITEM_ID_FIELD"], config["TIME_FIELD"]


def split_inputs(loader) -> dict:
    """History ids (n, 20), history length (n,), true next cluster (n,) of one split."""
    d = loader.dataset.inter_feat
    return {"seq": d[ITEM_SEQ], "length": d[ITEM_SEQ_LEN], "y": d[IID].numpy()}


eval_inputs = {"valid": split_inputs(valid_data), "test": split_inputs(test_data)}
splits = {}
for name, loader in (("train", train_data), ("valid", valid_data), ("test", test_data)):
    d = loader.dataset.inter_feat
    ts = d[TIME].numpy()
    splits[name] = {
        "targets": len(ts),
        "first_target_unix": int(ts.min()),
        "last_target_unix": int(ts.max()),
        "mean_history_len": round(float(d[ITEM_SEQ_LEN].float().mean()), 3),
    }
print(f"data build {t_data:.1f}s; users {dataset.user_num - 1}, clusters {dataset.item_num - 1}")
print(json.dumps(splits, indent=1))

data build 37.5s; users 813985, clusters 100
{
 "train": {
  "targets": 1749368,
  "first_target_unix": 1357518976,
  "last_target_unix": 1412776704,
  "mean_history_len": 5.853
 },
 "valid": {
  "targets": 218670,
  "first_target_unix": 1412776704,
  "last_target_unix": 1416398080,
  "mean_history_len": 7.391
 },
 "test": {
  "targets": 218670,
  "first_target_unix": 1416398080,
  "last_target_unix": 1420070400,
  "mean_history_len": 7.255
 }
}


## 2. Mô hình và optimizer

Cùng các lựa chọn mà `Trainer` của RecBole dùng cho config này: Adam, `learning_rate`, `weight_decay` (0), không cắt gradient. Loss = cross-entropy trên toàn bộ cluster (`loss_type: CE`), tính bởi `SMLPREC.calculate_loss`.

In [3]:
model = SMLPREC(config, train_data.dataset).to(config["device"])
optimizer = torch.optim.Adam(
    model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("parameters:", n_params)

SMLPREC(
  (item_embedding): Embedding(101, 64, padding_idx=0)
  (layers): ModuleList(
    (0): SMLP(
      (c): Sequential(
        (0): Linear(in_features=1, out_features=3, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=3, out_features=1, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (h): Sequential(
        (0): Linear(in_features=20, out_features=60, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=60, out_features=20, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (w): Sequential(
        (0): Linear(in_features=64, out_features=192, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=192, out_features=64, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((64,), eps=1e-05, el

## 3. Hàm huấn luyện, chấm điểm và đánh giá

- `train_epoch`: một lượt qua các mục tiêu train. Train loader của RecBole xáo trộn lại các dòng ở đầu mỗi lượt (`torch.randperm`).
- `full_scores`: điểm của mô hình cho mọi dòng trên toàn bộ item id; cột 0 là padding id của RecBole và được đặt -inf để không bao giờ được xếp hạng.
- `evaluate`: thứ hạng của cluster đúng trong mỗi dòng, rồi Recall@K / NDCG@K.

In [4]:
def train_epoch(model, loader, optimizer) -> float:
    """One epoch over the train loader; returns the sum of batch losses (RecBole's logged value)."""
    model.train()
    total = 0.0
    for inter in loader:
        inter = inter.to(config["device"])
        optimizer.zero_grad()
        loss = model.calculate_loss(inter)
        if torch.isnan(loss):
            raise ValueError("training loss is nan")
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total


@torch.no_grad()
def full_scores(model, inp: dict, batch: int = 8192) -> np.ndarray:
    """(n, n_items) scores for all rows of one split; padding column = -inf."""
    model.eval()
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: inp["seq"][i : i + batch], model.ITEM_SEQ_LEN: inp["length"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        out.append(s.cpu().numpy())
    return np.concatenate(out)


def evaluate(model, inp: dict) -> tuple[dict, np.ndarray]:
    """Recall@K / NDCG@K summary and the per-row rank of the true cluster."""
    rank = target_rank(full_scores(model, inp), inp["y"])
    return summarize(rank, ks=KS), rank

## 4. Huấn luyện

Valid và test được chấm sau mỗi epoch (test chỉ để ghi lại). Epoch tốt nhất được chọn theo **NDCG@10 trên valid** (`valid_metric` trong config), chỉ khi cải thiện nghiêm ngặt, đúng như quy tắc early-stopping của RecBole. Trọng số mỗi epoch được lưu để notebook 02 dùng lại mà không cần huấn luyện lại.

In [5]:
per_epoch, best = [], None
t1 = time.time()
for epoch in range(1, config["epochs"] + 1):
    t = time.time()
    loss = train_epoch(model, train_data, optimizer)
    row = {"epoch": epoch, "train_loss": round(loss, 4), "train_seconds": round(time.time() - t, 1)}
    ranks = {}
    for name, inp in eval_inputs.items():
        row[name], ranks[name] = evaluate(model, inp)
    valid_score = ndcg_at_k(ranks["valid"], 10)
    torch.save(model.state_dict(), RUN_DIR / f"epoch{epoch}.pth")
    if best is None or valid_score > best["valid_score"]:
        best = {"epoch": epoch, "valid_score": valid_score}
    per_epoch.append(row)
    print(
        f"epoch {epoch}: loss {row['train_loss']:.4f} ({row['train_seconds']}s) | "
        f"valid R@5 {row['valid']['recall@5']:.4f} NDCG@10 {row['valid']['ndcg@10']:.4f} | "
        f"test R@5 {row['test']['recall@5']:.4f} NDCG@10 {row['test']['ndcg@10']:.4f}"
    )
t_fit = time.time() - t1
print(f"train + per-epoch eval {t_fit:.1f}s; best epoch by valid NDCG@10: {best['epoch']}")

epoch 1: loss 6686.1206 (208.3s) | valid R@5 0.3325 NDCG@10 0.2819 | test R@5 0.3350 NDCG@10 0.2836


epoch 2: loss 6631.3986 (241.4s) | valid R@5 0.3347 NDCG@10 0.2834 | test R@5 0.3359 NDCG@10 0.2842


epoch 3: loss 6626.1111 (399.1s) | valid R@5 0.3352 NDCG@10 0.2838 | test R@5 0.3369 NDCG@10 0.2850
train + per-epoch eval 899.2s; best epoch by valid NDCG@10: 3


## 5. Test epoch tốt nhất

Nạp lại trọng số của epoch tốt nhất và chấm test đúng một lần. Checkpoint cũng được lưu theo định dạng của RecBole (`config` + `state_dict`) để `scripts/recommend_smlprec_expedia.py --checkpoint` nạp được.

In [6]:
state = torch.load(RUN_DIR / f"epoch{best['epoch']}.pth")
model.load_state_dict(state)
torch.save({"config": config, "epoch": best["epoch"] - 1, "state_dict": state}, RUN_DIR / "best.pth")
t2 = time.time()
valid_best, _ = evaluate(model, eval_inputs["valid"])
test_best, test_rank = evaluate(model, eval_inputs["test"])
t_test = time.time() - t2
print("valid:", valid_best)
print("test: ", test_best)

valid: {'n': 218670, 'recall@5': 0.3352, 'recall@10': 0.4623, 'recall@20': 0.6309, 'ndcg@5': 0.2429, 'ndcg@10': 0.2838, 'ndcg@20': 0.3263}
test:  {'n': 218670, 'recall@5': 0.3369, 'recall@10': 0.4645, 'recall@20': 0.6309, 'ndcg@5': 0.2439, 'ndcg@10': 0.285, 'ndcg@20': 0.3269}


## 6. Heuristic trên cùng các dòng test

- **Global popularity**: các cluster xếp theo số lần là mục tiêu train.
- **Repeat-last rồi popularity**: cluster user đặt gần nhất đứng đầu, phần còn lại theo độ phổ biến.

In [7]:
n_items = dataset.item_num
pop = np.bincount(train_data.dataset.inter_feat[IID].numpy(), minlength=n_items).astype(np.float64)
pop[0] = -np.inf  # padding id
y = eval_inputs["test"]["y"]
seq, length = eval_inputs["test"]["seq"].numpy(), eval_inputs["test"]["length"].numpy()
last = seq[np.arange(len(y)), length - 1]

scores_pop = np.tile(pop, (len(y), 1))
scores_rep = scores_pop.copy()
scores_rep[np.arange(len(y)), last] = pop.max() + 1
heuristics = {
    "global_popularity": summarize(target_rank(scores_pop, y), ks=KS),
    "repeat_last_then_popularity": summarize(target_rank(scores_rep, y), ks=KS),
}
for k, v in heuristics.items():
    print(f"{k:30s} {v}")

global_popularity              {'n': 218670, 'recall@5': 0.1478, 'recall@10': 0.2476, 'recall@20': 0.3925, 'ndcg@5': 0.0938, 'ndcg@10': 0.1256, 'ndcg@20': 0.1619}
repeat_last_then_popularity    {'n': 218670, 'recall@5': 0.257, 'recall@10': 0.3439, 'recall@20': 0.4762, 'ndcg@5': 0.2031, 'ndcg@10': 0.2309, 'ndcg@20': 0.2641}


## 7. Lưu `metrics.json`

In [8]:
commit = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False
).stdout.strip()
result = {
    "run": "week3 pipeline validation, 3 epochs - SMLP4Rec on Expedia bookings (research adaptation), notebook re-run",
    "dataset": "expedia (train.csv, is_booking == 1; item = hotel_cluster)",
    "git_commit": commit,
    "config": {
        k: config[k]
        for k in (
            "MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob", "loss_type",
            "epochs", "train_batch_size", "learning_rate", "seed", "eval_args", "topk", "valid_metric",
        )
    },
    "dataset_stats": {
        "users": int(dataset.user_num - 1),
        "items": int(dataset.item_num - 1),
        "sequence_targets": len(dataset.inter_feat),
    },
    "n_parameters": int(n_params),
    "splits": splits,
    "timing_seconds": {"data_build": round(t_data, 1), "train_and_eval": round(t_fit, 1), "test": round(t_test, 1)},
    "best_epoch_by_valid": best["epoch"],
    "checkpoint": str(RUN_DIR / "best.pth"),
    "per_epoch": per_epoch,
    "valid": valid_best,
    "test": test_best,
    "heuristics_same_test_rows": heuristics,
}
(RUN_DIR / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print("wrote", RUN_DIR / "metrics.json")

wrote C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-05_expedia_smlp4rec_week3-repro\metrics.json


## 8. Kiểm tra tái lập so với lượt chạy của script

Mọi giá trị số có ở cả JSON của script và kết quả của notebook này (kích thước các tập chia, train loss mỗi epoch, Recall@K và NDCG@K mỗi epoch và ở epoch tốt nhất, recall của heuristic). Thời gian chạy (wall-clock) được bỏ qua. Script làm tròn đến 4 chữ số, nên chạy lại chính xác sẽ cho `diff = 0`.

In [9]:
import pandas as pd  # after torch (see the first cell)

reference = json.loads(REFERENCE.read_text(encoding="utf-8"))
cmp = pd.DataFrame(compare_nested(reference, result), columns=["path", "script", "notebook", "diff"])
cmp = cmp[~cmp.path.str.startswith(("config.", "eval_args"))]
pd.set_option("display.max_rows", 200)
print(f"compared {len(cmp)} values; identical: {(cmp['diff'].abs() < 1e-9).sum()}; max |diff|: {cmp['diff'].abs().max():.6f}")
cmp

compared 77 values; identical: 77; max |diff|: 0.000000


,path,script,notebook,diff
14,dataset_stats.users,8.139850e+05,8.139850e+05,0.0
15,dataset_stats.items,1.000000e+02,1.000000e+02,0.0
16,dataset_stats.sequence_targets,2.186708e+06,2.186708e+06,0.0
17,n_parameters,3.370200e+04,3.370200e+04,0.0
18,splits.train.targets,1.749368e+06,1.749368e+06,0.0
19,splits.train.first_target_unix,1.357519e+09,1.357519e+09,0.0
20,splits.train.last_target_unix,1.412777e+09,1.412777e+09,0.0
21,splits.train.mean_history_len,5.853000e+00,5.853000e+00,0.0
22,splits.valid.targets,2.186700e+05,2.186700e+05,0.0
23,splits.valid.first_target_unix,1.412777e+09,1.412777e+09,0.0


In [10]:
# Persist the heuristics with NDCG (the script-era JSON only has Recall@K); the other keys of the
# reference file (top-5 behaviour, target mix) stay as they are.
reference["heuristics_same_test_rows"] = heuristics
REFERENCE.write_text(json.dumps(reference, indent=2), encoding="utf-8")
print("updated heuristics in", REFERENCE)

updated heuristics in C:\workspace\PROJECTS\next-best-product-recommendation\results\week3_implementation\smlprec_expedia_run.json
